<a href="https://colab.research.google.com/github/DrewThomasson/E2A-SML/blob/main/Notebooks/colab_e2a_sml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2A-SML on Google Colab

Select a GPU in **Runtime → Change runtime type**, then run the cell below. It installs E2A-SML and shows a link to the web GUI. Keep the cell running while you use the GUI. The **small** BookNLP model is a good starting point on the free tier.

The GUI link is public while this runtime is active. [Free Colab runtimes may end web UI sessions](https://research.google.com/colaboratory/faq.html); the optional CLI cell below works in the notebook. Download your SML file before the runtime ends.


In [ ]:
# @title 🚀 Run E2A-SML!
launch_web_gui = True # @param {type:"boolean"}

import os
import re
import shutil
import subprocess
import sys
import time
from collections import deque
from pathlib import Path
from queue import Empty, Queue
from threading import Thread
from IPython.display import clear_output

repo = Path(os.environ.get("E2A_SML_DIR", "/content/E2A-SML"))
ref = os.environ.get("E2A_SML_REF", "main")
log_path = repo.parent / "e2a_sml_colab.log"
activity = deque(maxlen=8)


def show_status(message):
    if not activity or activity[-1] != message:
        activity.append(message)
    clear_output(wait=True)
    print("🚀 E2A-SML\n")
    for item in activity:
        print(item)
    print(f"\n📝 Full log: {log_path}", flush=True)


def run_step(message, command, **kwargs):
    show_status(message)
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, **kwargs)
        try:
            while True:
                try:
                    code = process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    show_status(message + " (still working…)")
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        tail = "".join(log_path.read_text(errors="replace").splitlines(keepends=True)[-30:])
        raise RuntimeError(f"{message} failed (exit {code}).\n{tail}")


if not (repo / ".git").exists():
    if repo.exists():
        raise RuntimeError(f"{repo} exists but is not a Git checkout")
    run_step("📚 Downloading E2A-SML…",
             ["git", "clone", "--depth", "1", "--branch", ref,
              "https://github.com/DrewThomasson/E2A-SML.git", str(repo)])
else:
    show_status("✅ E2A-SML is already downloaded.")

if not shutil.which("ebook-convert"):
    run_step("📦 Installing ebook conversion tools…", ["apt-get", "update", "-qq"])
    run_step("📦 Installing ebook conversion tools…", ["apt-get", "install", "-y", "-qq", "calibre"])

run_step("📦 Installing uv…", [sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
if not Path(uv).exists():
    raise RuntimeError("uv was installed but could not be found")
venv_python = repo / ".venv" / "bin" / "python"
if not venv_python.exists():
    run_step("🐍 Preparing Python 3.10…", [uv, "venv", "--python", "3.10", str(repo / ".venv")])
run_step("📦 Installing E2A-SML packages…",
         [uv, "pip", "install", "--python", str(venv_python), "-r", str(repo / "requirements.txt")])
if subprocess.run([str(venv_python), "-c", "import en_core_web_sm"],
                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
    model_url = subprocess.check_output(
        [str(venv_python), "-m", "spacy", "info", "en_core_web_sm", "--url"], text=True
    ).strip()
    run_step("📦 Installing the English language model…",
             [uv, "pip", "install", "--python", str(venv_python), model_url])
run_step("🔎 Checking BookNLP resources…",
         [str(venv_python), "-c",
          "from sml_extractor.core import check_booknlp_installation; "
          "ok, message = check_booknlp_installation(); print(message); assert ok, message"], cwd=repo)
run_step("🎮 Checking the runtime GPU…",
         [str(venv_python), "-c",
          "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (CPU mode)')"])

if not launch_web_gui:
    show_status("✅ Setup is ready. Run the optional CLI cell below to process a book.")
else:
    show_status("🌐 Starting the web GUI…")
    gui = subprocess.Popen(
        [str(venv_python), "-u", "cli.py", "--gui", "--host", "0.0.0.0", "--share"],
        cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, "SML_DATA_DIR": str(repo / "data")},
    )
    lines = Queue()

    def read_gui():
        for line in gui.stdout:
            lines.put(line)
        lines.put(None)

    Thread(target=read_gui, daemon=True).start()
    recent = deque(maxlen=30)
    public_url = None
    try:
        with log_path.open("a", encoding="utf-8") as log:
            while True:
                try:
                    line = lines.get(timeout=30)
                except Empty:
                    if gui.poll() is not None:
                        break
                    if not public_url:
                        show_status("⏳ The web GUI is still starting…")
                    continue
                if line is None:
                    break
                log.write(line)
                log.flush()
                recent.append(line)
                match = re.search(r"https://[a-z0-9-]+\.gradio\.live", line)
                if match and not public_url:
                    public_url = match.group(0)
                    show_status(f"🎉 Ready! Open the web GUI: {public_url}\n📌 Keep this cell running while you use it.")
                elif public_url and any(word in line.lower() for word in ("error", "failed", "traceback")):
                    show_status(f"⚠️ {line.strip()}")
    except KeyboardInterrupt:
        gui.terminate()
    finally:
        if gui.poll() is None:
            gui.terminate()
        gui.wait()
        gui.stdout.close()
    if not public_url:
        raise RuntimeError("Web GUI did not produce a share link. Recent log:\n" + "".join(recent))
    show_status("👋 The web GUI stopped. Run this cell again to restart it.")


## Optional CLI

For CLI use, uncheck `launch_web_gui` in the setup cell, run it, then use the cell below to upload a book and download its SML file.


In [ ]:
#@title Optional: process a book through the CLI and download its SML file
model_size = "small" #@param ["small", "big"]
from google.colab import files
from pathlib import Path
import os
import subprocess

repo = Path(os.environ.get("E2A_SML_DIR", "/content/E2A-SML"))
venv_python = repo / ".venv" / "bin" / "python"
if not venv_python.exists():
    raise RuntimeError("Run the install cell first (you can uncheck launch_web_gui).")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No book was uploaded")
book = Path.cwd() / next(iter(uploaded))
output_dir = repo / "output"
subprocess.run([str(venv_python), "cli.py", str(book), "--model", model_size,
                "--output-dir", str(output_dir)], cwd=repo,
               env={**os.environ, "SML_DATA_DIR": str(repo / "data")}, check=True)
files.download(str(output_dir / (book.stem + ".e2a.sml.txt")))
